# XGBoost — Grade-Based Model (includes G1, G2)

This notebook trains an XGBoost model that **includes** the interim grades G1 and G2 as features.

Compared to the holistic model in `04_XG_Boost.ipynb` (R² ≈ 0.29), this model achieves a significantly higher R² because past exam performance is the strongest predictor of future performance.

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle

In [2]:
df = pd.read_csv("../data/student-mat.csv", sep=";")

print(df.shape)
print(df.columns)

(395, 33)
Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3'],
      dtype='object')


In [3]:
# Encode categorical columns using LabelEncoder
le_dict = {}

for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

In [4]:
# INCLUDE G1 and G2 as features (grade-based model)
X = df.drop(["G3"], axis=1)
y = df["G3"]

print(f"Features: {X.shape[1]}")
print(f"Feature list: {X.columns.tolist()}")

Features: 32
Feature list: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
model = XGBRegressor(n_estimators=100, learning_rate=0.1)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [7]:
y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

MAE: 1.1959185600280762
RMSE: 2.151055866283874
R2: 0.7743462324142456


In [8]:
# Save the grade-based model and encoders
pickle.dump(model, open("../xgb_model.pkl", "wb"))
pickle.dump(le_dict, open("../encoders.pkl", "wb"))

feature_columns = X.columns.tolist()
pickle.dump(feature_columns, open("../features.pkl", "wb"))

print("Model saved successfully!")
print(f"Features saved: {feature_columns}")

Model saved successfully!
Features saved: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']


## Prediction Demo

Test the grade-based model with sample student inputs — includes G1 and G2 grades.

In [9]:
def predict_student_grade_based(student_data):
    """Predict G3 using the grade-based model (includes G1, G2)."""
    
    input_df = pd.DataFrame([student_data])
    
    # Encode categoricals using the saved encoders
    for col in le_dict:
        if col in input_df.columns:
            input_df[col] = le_dict[col].transform(input_df[col])
    
    # Align with training features
    input_df = input_df.reindex(columns=X.columns, fill_value=0)
    
    prediction = model.predict(input_df)
    return round(prediction[0], 2)

In [10]:
# -------- Student 1: Strong student, high G1/G2 --------
student_1 = {
    "school": "GP", "sex": "M", "age": 18, "address": "U",
    "famsize": "GT3", "Pstatus": "T", "Medu": 4, "Fedu": 4,
    "Mjob": "teacher", "Fjob": "services", "reason": "course",
    "guardian": "mother", "traveltime": 1, "studytime": 3,
    "failures": 0, "schoolsup": "no", "famsup": "yes",
    "paid": "no", "activities": "yes", "nursery": "yes",
    "higher": "yes", "internet": "yes", "romantic": "no",
    "famrel": 4, "freetime": 3, "goout": 2, "Dalc": 1,
    "Walc": 1, "health": 4, "absences": 2,
    "G1": 15, "G2": 16
}

pred = predict_student_grade_based(student_1)
print(f"Student 1 (Strong, G1=15, G2=16) → Predicted G3: {pred}")

Student 1 (Strong, G1=15, G2=16) → Predicted G3: 16.739999771118164


In [11]:
# -------- Student 2: Average student --------
student_2 = {
    "school": "GP", "sex": "F", "age": 17, "address": "U",
    "famsize": "GT3", "Pstatus": "T", "Medu": 3, "Fedu": 3,
    "Mjob": "services", "Fjob": "other", "reason": "home",
    "guardian": "mother", "traveltime": 2, "studytime": 2,
    "failures": 0, "schoolsup": "no", "famsup": "yes",
    "paid": "no", "activities": "no", "nursery": "yes",
    "higher": "yes", "internet": "yes", "romantic": "no",
    "famrel": 4, "freetime": 3, "goout": 3, "Dalc": 1,
    "Walc": 2, "health": 3, "absences": 6,
    "G1": 10, "G2": 11
}

pred = predict_student_grade_based(student_2)
print(f"Student 2 (Average, G1=10, G2=11) → Predicted G3: {pred}")

Student 2 (Average, G1=10, G2=11) → Predicted G3: 10.510000228881836


In [12]:
# -------- Student 3: At-risk student, low G1/G2 --------
student_3 = {
    "school": "MS", "sex": "M", "age": 19, "address": "R",
    "famsize": "GT3", "Pstatus": "A", "Medu": 1, "Fedu": 1,
    "Mjob": "other", "Fjob": "other", "reason": "other",
    "guardian": "other", "traveltime": 3, "studytime": 1,
    "failures": 2, "schoolsup": "yes", "famsup": "no",
    "paid": "no", "activities": "no", "nursery": "no",
    "higher": "no", "internet": "no", "romantic": "yes",
    "famrel": 2, "freetime": 5, "goout": 5, "Dalc": 3,
    "Walc": 4, "health": 2, "absences": 20,
    "G1": 5, "G2": 4
}

pred = predict_student_grade_based(student_3)
print(f"Student 3 (At-risk, G1=5, G2=4) → Predicted G3: {pred}")

Student 3 (At-risk, G1=5, G2=4) → Predicted G3: 6.730000019073486


In [13]:
# -------- Student 4: Improving student (low G1, higher G2) --------
student_4 = {
    "school": "GP", "sex": "F", "age": 16, "address": "U",
    "famsize": "LE3", "Pstatus": "T", "Medu": 3, "Fedu": 2,
    "Mjob": "health", "Fjob": "services", "reason": "reputation",
    "guardian": "mother", "traveltime": 1, "studytime": 4,
    "failures": 1, "schoolsup": "yes", "famsup": "yes",
    "paid": "yes", "activities": "yes", "nursery": "yes",
    "higher": "yes", "internet": "yes", "romantic": "no",
    "famrel": 5, "freetime": 2, "goout": 2, "Dalc": 1,
    "Walc": 1, "health": 5, "absences": 0,
    "G1": 7, "G2": 12
}

pred = predict_student_grade_based(student_4)
print(f"Student 4 (Improving, G1=7, G2=12) → Predicted G3: {pred}")

Student 4 (Improving, G1=7, G2=12) → Predicted G3: 11.65999984741211
